In [1]:
import sys
sys.path.insert(0, '../')

import pandas as pd

from automed import *
from IPython.display import display, Markdown as IMarkdown
from rich.console import Console
from rich.markdown import Markdown

[16:33:25] cuDF not found: falling back to standalone pandas.

In [2]:
titanic = pd.read_csv('../perf_logger/tests_data/titanic.csv', delimiter=';')

In [3]:
autom = AutoMed()

print(autom.default_pipeline())
print(autom.json_pipeline())

None
{'step': 'MetaOrderedStep', 'name': 'MetaStep', 'description': 'Step description...', 'configuration': {}, 'children': [{'step': 'MetaStep', 'name': 'MetaStep', 'description': 'Step description...', 'configuration': {}, 'children': [{'step': 'ActDropTextualColumn', 'name': 'Drop textual columns', 'description': 'Drop textual columns.', 'configuration': {}, 'children': []}, {'step': 'ActTfIdf', 'name': 'TF-IDF', 'description': 'Step description...', 'configuration': {}, 'children': []}, {'step': 'ActSplitDate', 'name': 'Transform string column to date', 'description': 'Step description...', 'configuration': {}, 'children': []}, {'step': 'ActMeanColumn', 'name': 'Fill missing values with mean', 'description': 'Fills missing values with the mean of non-missing values\n        when the proportion of empty rows is lower than {empty_threshold}.', 'configuration': {'empty_threshold': {'description': 'Column with less or equal proportion of empty row will\n                    be fill with

In [4]:
pipeline = {
    'step': 'MetaOrderedStep',
    'children': [
        {
            'step': 'ActDropNumericalColumn',
            'configuration': {
                'empty_threshold': { 'value': 0.1 },
            }
        },
        {
            'step': 'MetaStep',
            'tag': 'cleaning',
        },
        {
            'step': 'MetaStep',
            'tag': 'features_selection',
        },
        {
            'step': 'WrapKFold',
            'children': [{
                'step': 'ActKNN',
            }]
        },
    ]
}

autom.load_pipeline(pipeline)
print(autom.json_pipeline())


{'step': 'MetaOrderedStep', 'name': 'MetaStep', 'description': 'Step description...', 'configuration': {}, 'children': [{'step': 'ActDropNumericalColumn', 'name': 'Drop numerical columns', 'description': 'Drop numerical columns where the proportion of empty rows\n        in the dataset is higher than {empty_threshold}.', 'configuration': {'empty_threshold': {'description': 'Column with more or equal proportion of empty row                     will dropped. 1 will drop all columns', 'default': 0.5, 'value': 0.1}}, 'children': []}, {'step': 'MetaStep', 'name': 'MetaStep', 'description': 'Step description...', 'configuration': {}, 'children': [{'step': 'ActDropTextualColumn', 'name': 'Drop textual columns', 'description': 'Drop textual columns.', 'configuration': {}, 'children': []}, {'step': 'ActTfIdf', 'name': 'TF-IDF', 'description': 'Step description...', 'configuration': {}, 'children': []}, {'step': 'ActSplitDate', 'name': 'Transform string column to date', 'description': 'Step desc

In [5]:
results = autom.fit(
    titanic.drop('label', axis=1).copy(),
    titanic[['label']]).copy()

Output()

           running step: MetaOrderedStep (steps=ActDropNumericalColumn,WrapKFold,MetaStep)

[16:33:26] running step: MetaStep                                                                                  
           (steps=ActDropNumericalColumn,ActMeanColumn,ActTfIdf,ActSplitDate,ActDropDateColumn,ActDropTextualColumn
           ,ActOnehot)

[16:33:28] running step: MetaStep (steps=ActRemoveHighCorrelatedColumn)

[16:33:37] running step: WrapKFold (step=ActKNN, folds=5, stratify=True)

...

c:\Users\0869778\dev\automl\.venv\Lib\site-packages\sklearn\neighbors\_classification.py:238: 
DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to 
(n_samples,), for example using ravel().
  return self._fit(X, y)

c:\Users\0869778\dev\automl\.venv\Lib\site-packages\sklearn\neighbors\_classification.py:238: 
DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to 
(n_samples,), for example using ravel().
  return self._fit(X, y)

c:\Users\0869778\dev\automl\.venv\Lib\site-packages\sklearn\neighbors\_classification.py:238: 
DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to 
(n_samples,), for example using ravel().
  return self._fit(X, y)

c:\Users\0869778\dev\automl\.venv\Lib\site-packages\sklearn\neighbors\_classification.py:238: 
DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to 
(n_samples,), for example using ravel().
  return self._fit(X, y)

c:\Users\0869778\dev\automl\.venv\Lib\site-packages\sklearn\neighbors\_classification.py:238: 
DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to 
(n_samples,), for example using ravel().
  return self._fit(X, y)

In [6]:
print(results[0].pipeline.model)
print(results[0].pipeline.steps)

console = Console()

for step in results[0].pipeline.explanations:
    md = step.to_markdown()
    if md:
        # console.print(Markdown(md))
        display(IMarkdown(md))

pm = results[0].pipeline.pickle()

Learn : KNN
[('Drop numerical columns', <automed.actionables.cleaning.act_drop_numerical_column.ActDropNumericalColumn object at 0x000001ED0E097910>), ('Fill missing values with mean', <automed.actionables.cleaning.act_mean_column.ActMeanColumn object at 0x000001ED0FA68750>), ('Transform string column to date', <automed.actionables.cleaning.act_split_date.ActSplitDate object at 0x000001ED0FA5B990>), ('One hot encoding categorical features', <automed.actionables.cleaning.act_onehot.ActOnehot object at 0x000001ED0FA68A10>), ('TF-IDF', <automed.actionables.cleaning.act_tf_idf.ActTfIdf object at 0x000001ED0FA68790>), ('Drop textual columns', <automed.actionables.cleaning.act_drop_textual_column.ActDropTextualColumn object at 0x000001ED0FA685D0>), ('Drop date columns', <automed.actionables.cleaning.act_drop_date_column.ActDropDateColumn object at 0x000001ED0FA68850>), ('Drop numerical columns', <automed.actionables.cleaning.act_drop_numerical_column.ActDropNumericalColumn object at 0x000001


## Drop numerical columns
**Drop numerical columns where the proportion of empty rows
        in the dataset is higher than 0.1.**


### Configuration
| Name | Description | Value |
| ---- | ----------- | ----- |
| **empty_threshold** | Column with more or equal proportion of empty row                     will dropped. 1 will drop all columns | 0.1 |



### Processings
 - Dropped column **`Age`** because **177** values out of
                **891** (**19.87%**) are empty.




        


## One hot encoding categorical features
**Step description...**




### Processings
 - Encoded categorical column **`Sex`** into **2** new columns.
 - Encoded categorical column **`Embarked`** into **4** new columns.




        


## TF-IDF
**Step description...**




### Processings
 - Encoded text column **`Name`** into **1509** new columns.
 - Encoded text column **`Ticket`** into **695** new columns.
 - Encoded text column **`Cabin`** into **158** new columns.




        


## Remove High Correlated Column
**Remove columns which correlation with other columns is higher than 0.9.**


### Configuration
| Name | Description | Value |
| ---- | ----------- | ----- |
| **threshold** | If two columns is correlated over this value, only one                     will be kept | 0.9 |



### Processings
 - Dropped column **`Sex_male`** because it was too correlated with
                **`Sex_female`**.
 - Dropped column **`Name_amanda`** because it was too correlated with
                **`Name_adolfina`**.
 - Dropped column **`Name_antino`** because it was too correlated with
                **`Name_aijo`**.
 - Dropped column **`Name_asim`** because it was too correlated with
                **`Name_adola`**.
 - Dropped column **`Name_aurora`** because it was too correlated with
                **`Name_adelia`**.
 - Dropped column **`Name_banoura`** because it was too correlated with
                **`Name_ayoub`**.
 - Dropped column **`Name_barah`** because it was too correlated with
                **`Name_assi`**.
 - Dropped column **`Name_barkworth`** because it was too correlated with
                **`Name_algernon`**.
 - Dropped column **`Name_bazzani`** because it was too correlated with
                **`Name_albina`**.
 - Dropped column **`Name_berthe`** because it was too correlated with
                **`Name_antonine`**.
 - Dropped column **`Name_blyler`** because it was too correlated with
                **`Name_billiard`**.
 - Dropped column **`Name_bratthammer`** because it was too correlated with
                **`Name_bernt`**.
 - Dropped column **`Name_butt`** because it was too correlated with
                **`Name_archibald`**.
 - Dropped column **`Name_cassem`** because it was too correlated with
                **`Name_albimona`**.
 - Dropped column **`Name_cerin`** because it was too correlated with
                **`Name_balkic`**.
 - Dropped column **`Name_chip`** because it was too correlated with
                **`Name_chang`**.
 - Dropped column **`Name_christine`** because it was too correlated with
                **`Name_carla`**.
 - Dropped column **`Name_chronopoulos`** because it was too correlated with
                **`Name_apostolos`**.
 - Dropped column **`Name_clear`** because it was too correlated with
                **`Name_cameron`**.
 - Dropped column **`Name_cornelia`** because it was too correlated with
                **`Name_alma`**.
 - *and **1102** more processings...*



        


## K-Fold cross validation
**Performs cross-validation on the dataset, splitting into
        train and test sets 5 times.**


### Configuration
| Name | Description | Value |
| ---- | ----------- | ----- |
| **folds** | Split ratio | 5 |
| **stratify** | Whether to run stratified K-Fold | True |



### Processings
 - Computed mean metrics.
 - Trained 5 models, then one last model
                on the whole dataset, and returned it as the output.




        


## Learn : KNN
**Step description...**


### Configuration
| Name | Description | Value |
| ---- | ----------- | ----- |
| **metric** | Can be minkowski or manhattan | minkowski |
| **n_neighbors** | Number of neighbors | 5 |





### Metrics
| Metric name | Computed value | Description |
| ----------- | -------------- | ----------- |
| `accuracy` | **0.5442** | *Accuracy classification score.* |
| `balanced_accuracy` | **0.5167** | *Compute the balanced accuracy.* |
| `classification_error` | **0.4833** | *Computes the classification error, if accuracy is relevant then the             classification error is calculated by 1 - accuracy otherwise if balanced             accuracy is relevant then the classification error is             calculated by 1 - balanced _accuracy.* |
| `f1_score` | **0.4800** | *Compute the F1 score, also known as balanced F-score or F-measure.             The F1 score can be interpreted as a harmonic mean of the precision and recall* |
| `precision` | **0.5818** | *Compute the precision: The precision is the ratio tp / (tp + fp)             where tp is the number of true positives and fp the number of false positives.* |
| `recall` | **0.5818** | *The recall is the ratio tp / (tp + fn) where tp is the number             of true positives and fn the number of false negatives. The recall is             intuitively the ability of the classifier to find all the positive samples.* |
| `specificity` | **0.6400** | *The proportion of negative instances that are corectely             classified as negative: tn / (tn + fp)* |

        

In [7]:
import pickle

o = 200 # offset
n = 68  # # of samples
labels = titanic.iloc[o:(o+n)]['label']
predict_df = titanic.iloc[o:(o+n)].drop('label', axis=1).copy()

# labels = labels.reset_index()
predict_df.reset_index(inplace=True, drop=True)

m = pickle.loads(pm)
sum([ r == labels[o+i] for i, r in enumerate(m.predict(predict_df)) ]) / n

0.7352941176470589

In [11]:
final_boss_automed = AutoMed(max_workers=2)
final_boss_automed.default_pipeline()
final_boss_results = final_boss_automed.fit(titanic.drop('label', axis=1).copy(), titanic[['label']].copy())

[10:53:18] running step: MetaOrderedStep (steps=MetaStep,MetaExplorerStep)

           running step: MetaStep                                                                                  
           (steps=ActSplitDate,ActDropTextualColumn,ActTfIdf,ActOnehot,ActDropDateColumn,ActMeanColumn,ActDropNumer
           icalColumn)

[10:53:20] running step: MetaStep (steps=ActRemoveHighCorrelatedColumn)

[10:53:28] running step: MetaStep (steps=ActRandomOverSampling,ActMinMaxScaler)

           running step: MetaExplorerStep (steps=WrapGeneticGridSearch)

...

           Traceback (most recent call last):                                                                      
             File "c:\Users\0869778\dev\automl\.venv\Lib\site-packages\pandas\core\indexes\base.py", line 3790, in 
           get_loc                                                                                                 
               return self._engine.get_loc(casted_key)                                                             
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^                                                             
             File "index.pyx", line 152, in pandas._libs.index.IndexEngine.get_loc                                 
             File "index.pyx", line 181, in pandas._libs.index.IndexEngine.get_loc                                 
             File "pandas\_libs\hashtable_class_helper.pxi", line 7080, in                                         
           pandas._libs.hashtable.PyObjectHashTable.get_item                                                       
             File "pandas\_libs\hashtable_class_helper.pxi", line 7088, in                                         
           pandas._libs.hashtable.PyObjectHashTable.get_item                                                       
           KeyError: 0                                                                                             
                                                                                                                   
           The above exception was the direct cause of the following exception:                                    
                                                                                                                   
           Traceback (most recent call last):                                                                      
             File "c:\Users\0869778\dev\automl\src\sandbox\..\automed\worker_manager.py", line 35, in run          
               return self.task(*self.args, **self.kwargs)                                                         
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^                                                         
             File "c:\Users\0869778\dev\automl\src\sandbox\..\automed\step.py", line 587, in runner_wrapper        
               output = func(self, current_input, callback=callback)                                               
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^                                               
             File "c:\Users\0869778\dev\automl\src\sandbox\..\automed\wrapper\dataset\wrap_kfold.py", line 63, in  
           run                                                                                                     
               metrics.append(output.evaluate(test_ds, force=True))                                                
                              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^                                                 
             File "c:\Users\0869778\dev\automl\src\sandbox\..\automed\output.py", line 218, in evaluate            
               result = dataset.compute_metric(self.pipeline, metric)                                              
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^                                              
             File "c:\Users\0869778\dev\automl\src\sandbox\..\automed\dataset.py", line 299, in compute_metric     
               return metric.compute(self.__y, y_pred)                                                             
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^                                                             
             File "c:\Users\0869778\dev\automl\src\sandbox\..\automed\metrics\f1_score_metric.py", line 56, in     
           compute                                                                                                 
               return f1_score(y, y_pred, pos_label=y[0]

           running step: WrapKFold (step=ActXGBoost, folds=5, stratify=True)

           running k-folds: ActXGBoost             (max_depth=14, random_state=42, learning_rate=0.552995971948809,
           n_estimators=206)

<class 'pandas.core.frame.DataFrame'>

<class 'pandas.core.frame.DataFrame'>

In [ ]:
# sum([ len(r.model.pickle()) for r in final_boss_results ])
[ (r.model.ml_model, r.evaluate()) for r in final_boss_results if r.model.ml_model is not None ]

NameError: name 'final_boss_results' is not defined